# Central Multilingual Transfer-Learning Translator — Google Colab (CPU-only)

This single notebook implements a **central multilingual translation model** that can be fine-tuned repeatedly as new language-pair datasets are added.

## Intended workflow
1. Select a learned language from the dropdown and enter text to translate.
2. Select **Add a language** to reveal the dataset/model upload and training controls immediately below the dropdown.
3. Upload a parallel CSV/XLSX dataset, or a compatible saved model bundle ZIP to continue from.
4. Fine-tune the **current central model** for a bounded CPU budget.
5. Evaluate on a held-out set, save the updated model and language registry, and download one ZIP bundle.
6. Re-upload that bundle in a future Colab session to resume inference or further training.

## Important constraints
- **No GPU is used.** A configurable training budget stops optimization after approximately 18 minutes. Downloads, tokenization, evaluation and saving can make total time exceed 20 minutes; no notebook can guarantee a universal wall-clock limit on variable Colab CPUs.
- Use a multilingual seq2seq checkpoint compatible with the languages and direction you need. The default is `facebook/nllb-200-distilled-600M`, a large model for CPU; for a more practical CPU experiment, replace it with a smaller multilingual seq2seq checkpoint that supports your target language pairs. Model availability, license, access, memory and latency vary.
- The notebook does not claim a language was learned merely because it was added to the registry. It registers the language only after the updated model is saved successfully.
- Training examples are not used as a translation lookup cache. Inference always calls the model's `generate()` function.
- A dataset needs two aligned text columns. Column names are auto-detected from common names; no manual source/target column fields are shown.
- An uploaded model ZIP must contain a compatible Transformers encoder-decoder model. A dataset is still required to teach a new language; a model ZIP supplies the starting weights, it is not itself training data. Unrelated architectures cannot be merged automatically.
- Fine-tuning one shared model can cause catastrophic forgetting. This notebook stores the central model as a new version and uses a held-out set, but a production research system should also retain replay examples from earlier languages and evaluate all previous directions after every update.

**Output:** a downloadable ZIP containing model weights, tokenizer/configuration, registry and metadata. The working files are stored under `/content/central_translator/` and are temporary unless downloaded or copied to persistent storage.

## CPU/RAM-safe defaults
Training defaults to 500 dataset rows, 64 tokens, batch size 1, gradient checkpointing, and Adafactor optimizer to reduce RAM use. Held-out generation is limited to 8 examples. Larger models/datasets can still exceed the RAM available in a particular Colab runtime; the 20-minute target is best-effort, not guaranteed.

## 1. Install dependencies

In [1]:
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "datasets", "sentencepiece", "safetensors",
    "accelerate", "gradio", "pandas", "matplotlib", "openpyxl", "sacrebleu"
])
print("Dependencies installed. If imports fail after package installation, restart the runtime and run all cells from the top.")

Dependencies installed. If imports fail after package installation, restart the runtime and run all cells from the top.


## 2. Imports and central-model workspace

In [2]:
import os, re, json, time, shutil, zipfile, traceback, tempfile
from pathlib import Path
from datetime import datetime, timezone

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gradio as gr
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments,
    Seq2SeqTrainer, TrainerCallback
)
from sacrebleu.metrics import BLEU, CHRF

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
ROOT = Path("/content/central_translator")
VERSIONS_DIR = ROOT / "versions"
ROOT.mkdir(parents=True, exist_ok=True)
VERSIONS_DIR.mkdir(parents=True, exist_ok=True)
REGISTRY_PATH = ROOT / "languages.json"
STATE_PATH = ROOT / "central_state.json"
METRICS_PATH = ROOT / "metrics.csv"
DEFAULT_MODEL_ID = "facebook/nllb-200-distilled-600M"

def slug(s):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(s)).strip("_")[:60] or "item"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def load_registry():
    try:
        obj = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
        if isinstance(obj, dict) and isinstance(obj.get("languages"), list):
            return obj
    except Exception:
        pass
    return {"schema_version": 1, "languages": [], "updated_at": None}

def save_registry(obj):
    obj["updated_at"] = utc_now()
    tmp = REGISTRY_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(REGISTRY_PATH)

def load_state():
    try:
        return json.loads(STATE_PATH.read_text(encoding="utf-8"))
    except Exception:
        return {"central_model_dir": None, "base_model_id": DEFAULT_MODEL_ID, "version": 0}

def save_state(state):
    tmp = STATE_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(STATE_PATH)

def language_choices():
    # Display translation directions explicitly so inference picks the correct pair.
    labels = []
    for item in load_registry()["languages"]:
        src = str(item.get("source_language", "")).strip()
        tgt = str(item.get("target_language", "")).strip()
        name = str(item.get("name", "")).strip()
        label = f"{src} → {tgt}" if src and tgt else name
        if label and label not in labels:
            labels.append(label)
    return labels + ["Add a language"]

def source_code_for(lang):
    # NLLB language codes; edit/extend this mapping for additional languages.
    mapping = {
        "english": "eng_Latn", "hindi": "hin_Deva", "marathi": "mar_Deva",
        "bengali": "ben_Beng", "tamil": "tam_Taml", "telugu": "tel_Telu",
        "gujarati": "guj_Gujr", "punjabi": "pan_Guru", "urdu": "urd_Arab",
        "french": "fra_Latn", "german": "deu_Latn", "spanish": "spa_Latn",
        "arabic": "arb_Arab", "chinese": "zho_Hans", "japanese": "jpn_Jpan",
        "korean": "kor_Hang", "portuguese": "por_Latn", "russian": "rus_Cyrl"
    }
    return mapping.get(str(lang).strip().lower(), "")

def load_central_model(model_dir=None, fallback_id=None):
    state = load_state()
    model_dir = model_dir or state.get("central_model_dir")
    if model_dir and Path(model_dir, "config.json").exists():
        tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(model_dir, local_files_only=True)
        model_source = str(model_dir)
    else:
        model_source = fallback_id or state.get("base_model_id") or DEFAULT_MODEL_ID
        tok = AutoTokenizer.from_pretrained(model_source)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(model_source)
    mdl.to("cpu").eval()
    return tok, mdl, model_source

print("CPU-only:", not torch.cuda.is_available(), "| workspace:", ROOT)
print("Registry languages:", [x.get("name") for x in load_registry()["languages"]])

CPU-only: True | workspace: /content/central_translator
Registry languages: []


## 3. Dataset and upload validation

In [3]:
def read_dataset(path):
    if not path:
        raise ValueError("Upload a CSV/XLSX parallel dataset, or select a model ZIP.")
    p = Path(path)
    if p.suffix.lower() == ".csv":
        df = pd.read_csv(p)
    elif p.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(p)
    else:
        raise ValueError("Dataset must be CSV or XLSX.")
    if df.empty:
        raise ValueError("Dataset has no rows.")
    return df

def auto_pair_columns(df):
    # Prefer known paired column names; otherwise use the first two text-like columns.
    lower = {str(c).strip().lower(): c for c in df.columns}
    pairs = [
        ("text_in_language", "translated_text"), ("source", "target"),
        ("source_text", "target_text"), ("original_text", "translated_text"),
        ("hindi_text", "english_text"), ("src", "tgt"),
        ("sentence", "translation"), ("text", "translation")
    ]
    for a, b in pairs:
        if a in lower and b in lower:
            return lower[a], lower[b]
    candidates = [c for c in df.columns if df[c].dtype == object]
    if len(candidates) >= 2:
        return candidates[0], candidates[1]
    if len(df.columns) >= 2:
        return df.columns[0], df.columns[1]
    raise ValueError(f"Need two columns containing parallel sentences. Columns found: {list(df.columns)}")

def clean_parallel_data(path, max_rows=2500):
    df = read_dataset(path)
    source_col, target_col = auto_pair_columns(df)
    df = df[[source_col, target_col]].dropna().copy()
    df[source_col] = df[source_col].astype(str).str.strip()
    df[target_col] = df[target_col].astype(str).str.strip()
    df = df[(df[source_col] != "") & (df[target_col] != "")]
    df = df.drop_duplicates(subset=[source_col, target_col]).reset_index(drop=True)
    if len(df) < 8:
        raise ValueError(f"Only {len(df)} usable parallel pairs found; at least 8 are required.")
    if len(df) > int(max_rows):
        df = df.sample(n=int(max_rows), random_state=42).reset_index(drop=True)
    return df, source_col, target_col

def validate_model_zip(zip_path, extract_dir):
    if not zip_path:
        raise ValueError("Choose a model ZIP file.")
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        if any(Path(n).is_absolute() or ".." in Path(n).parts for n in names):
            raise ValueError("Unsafe path detected in model ZIP.")
        zf.extractall(extract_dir)
    candidates = [extract_dir] + [p for p in extract_dir.iterdir() if p.is_dir()]
    for p in candidates:
        if (p / "config.json").exists():
            cfg = AutoConfig.from_pretrained(str(p), local_files_only=True)
            if not getattr(cfg, "is_encoder_decoder", False):
                raise ValueError("The uploaded model is not a compatible encoder-decoder translation model.")
            return p
    raise ValueError("Could not find config.json at the ZIP root or in its immediate model/ folder.")

class StopAtBudget(TrainerCallback):
    def __init__(self, seconds):
        self.seconds = max(60, int(seconds))
        self.start = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.monotonic()
        return control
    def on_step_end(self, args, state, control, **kwargs):
        if self.start is not None and time.monotonic() - self.start >= self.seconds:
            control.should_training_stop = True
        return control

print("Dataset and bundle validators ready.")

Dataset and bundle validators ready.


## 4. Central model training, evaluation, registry, and bundle export

In [4]:
def write_bundle(model_dir, registry, metadata, output_zip):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        raise FileNotFoundError("Model config missing; refusing to export incomplete bundle.")
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in model_dir.rglob("*"):
            if f.is_file():
                zf.write(f, arcname=str(Path("central_model") / f.relative_to(model_dir)))
        zf.writestr("languages.json", json.dumps(registry, ensure_ascii=False, indent=2))
        zf.writestr("central_state.json", json.dumps(metadata, ensure_ascii=False, indent=2))
    return str(output_zip)

def train_or_extend_central(file_path, language_name, source_language, target_language,
                            lang_code, max_rows=500, max_length=64,
                            epochs=1, budget_minutes=18, uploaded_model_zip=None):
    start_time = time.monotonic()
    trainer = None
    try:
        language_name = str(language_name or "").strip()
        source_language = str(source_language or "").strip()
        target_language = str(target_language or "").strip()
        if not language_name or not source_language or not target_language:
            raise ValueError("Enter language name, source language, and target language.")
        state = load_state()
        base_dir = state.get("central_model_dir")
        base_id = state.get("base_model_id") or DEFAULT_MODEL_ID

        # An uploaded compatible bundle/model is treated as the new central starting point.
        if uploaded_model_zip:
            extracted = ROOT / "incoming_model"
            if extracted.exists():
                shutil.rmtree(extracted)
            model_path = validate_model_zip(uploaded_model_zip, extracted)
            tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
            model = AutoModelForSeq2SeqLM.from_pretrained(str(model_path), local_files_only=True)
            base_description = str(model_path)
        elif base_dir and Path(base_dir, "config.json").exists():
            tokenizer = AutoTokenizer.from_pretrained(base_dir, local_files_only=True)
            model = AutoModelForSeq2SeqLM.from_pretrained(base_dir, local_files_only=True)
            base_description = str(base_dir)
        else:
            tokenizer, model = None, None
            base_description = base_id

        if file_path:
            df, src_col, tgt_col = clean_parallel_data(file_path, int(max_rows))
        else:
            raise ValueError("For training a new language, upload a parallel CSV/XLSX dataset. A model ZIP can initialize/continue training, but a dataset is still needed to adapt it.")
        if tokenizer is None or model is None:
            tokenizer = AutoTokenizer.from_pretrained(base_id)
            model = AutoModelForSeq2SeqLM.from_pretrained(base_id)
        if not getattr(model.config, "is_encoder_decoder", False):
            raise ValueError("The selected model is not a seq2seq encoder-decoder model.")
        model.to("cpu")
        # Reduce activation memory during CPU fine-tuning.
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False

        # NLLB requires explicit source language token and target forced BOS token.
        src_code = source_code_for(source_language)
        tgt_code = source_code_for(target_language)
        if "nllb" in str(getattr(model.config, "_name_or_path", "")).lower() or "nllb" in base_description.lower() or "nllb" in base_id.lower():
            if not src_code or not tgt_code:
                raise ValueError("For NLLB, the language names need an NLLB code mapping. Add the correct code in source_code_for().")
            tokenizer.src_lang = src_code
            forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_code)
            if forced_bos_token_id is None or forced_bos_token_id == tokenizer.unk_token_id:
                raise ValueError(f"Tokenizer does not recognize target language code {tgt_code}.")
        else:
            forced_bos_token_id = None

        # Random but reproducible split; no held-out examples used in gradient updates.
        shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
        n_test = max(1, int(round(len(shuffled) * 0.15)))
        test_df, train_df = shuffled.iloc[:n_test].reset_index(drop=True), shuffled.iloc[n_test:].reset_index(drop=True)

        def tokenize_batch(batch):
            # NLLB uses different language codes for encoder input and decoder targets.
            if src_code:
                tokenizer.src_lang = src_code
            inputs = tokenizer(batch["source"], max_length=int(max_length), truncation=True)
            if tgt_code and hasattr(tokenizer, "tgt_lang"):
                tokenizer.tgt_lang = tgt_code
            try:
                labels = tokenizer(text_target=batch["target"], max_length=int(max_length), truncation=True)
            except TypeError:
                with tokenizer.as_target_tokenizer():
                    labels = tokenizer(batch["target"], max_length=int(max_length), truncation=True)
            inputs["labels"] = labels["input_ids"]
            if src_code:
                tokenizer.src_lang = src_code
            return inputs

        # Tokenize serially in small batches; do not create a second tokenized test dataset.
        train_ds = Dataset.from_dict({"source": train_df[src_col].tolist(), "target": train_df[tgt_col].tolist()}).map(
            tokenize_batch, batched=True, batch_size=16, remove_columns=["source", "target"], load_from_cache_file=False
        )
        test_ds = None  # Keep held-out strings un-tokenized to save RAM.
        import gc
        gc.collect()

        version = int(state.get("version", 0)) + 1
        version_dir = VERSIONS_DIR / f"central_v{version}_{slug(language_name)}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        version_dir.mkdir(parents=True, exist_ok=False)
        args = Seq2SeqTrainingArguments(
            output_dir=str(version_dir / "trainer_tmp"),
            num_train_epochs=float(epochs),
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            learning_rate=2e-5,
            optim="adafactor",
            gradient_checkpointing=True,
            gradient_accumulation_steps=1,
            logging_strategy="steps", logging_steps=10,
            save_strategy="no", report_to=[],
            use_cpu=True, dataloader_num_workers=0, dataloader_pin_memory=False, fp16=False,
            remove_unused_columns=True
        )
        trainer = Seq2SeqTrainer(
            model=model, args=args, train_dataset=train_ds,
            data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
            processing_class=tokenizer,
            callbacks=[StopAtBudget(int(budget_minutes) * 60)]
        )
        trainer.train()
        # Release training-only activation/checkpoint state before CPU generation.
        if hasattr(trainer.model, "gradient_checkpointing_disable"):
            trainer.model.gradient_checkpointing_disable()
        if hasattr(trainer.model.config, "use_cache"):
            trainer.model.config.use_cache = True
        import gc
        gc.collect()

        # Save complete central model to a fresh version before registry/state update.
        model_dir = version_dir / "central_model"
        model_dir.mkdir(parents=True, exist_ok=True)
        trainer.model.save_pretrained(model_dir, safe_serialization=True)
        tokenizer.save_pretrained(model_dir)

        # Held-out inference; limit examples to keep CPU evaluation bounded.
        refs, hyps, rows = [], [], []
        eval_n = min(8, len(test_df))
        eval_model = trainer.model.to("cpu").eval()
        for _, row in test_df.head(eval_n).iterrows():
            src_text, ref_text = str(row[src_col]), str(row[tgt_col])
            encoded = tokenizer(src_text, return_tensors="pt", truncation=True, max_length=int(max_length))
            gen_kwargs = {"max_new_tokens": 96, "num_beams": 1}
            if forced_bos_token_id is not None:
                gen_kwargs["forced_bos_token_id"] = int(forced_bos_token_id)
            with torch.inference_mode():
                output = eval_model.generate(**encoded, **gen_kwargs)
            hyp = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
            refs.append(ref_text); hyps.append(hyp)
            if len(rows) < 8:
                rows.append({"source": src_text, "reference": ref_text, "generated": hyp})
        bleu = float(BLEU().corpus_score(hyps, [refs]).score) if hyps else 0.0
        chrf = float(CHRF().corpus_score(hyps, [refs]).score) if hyps else 0.0

        logs = trainer.state.log_history
        loss_path = version_dir / "training_loss.png"
        fig, ax = plt.subplots(figsize=(7, 3.5))
        points = [(x.get("step"), x.get("loss")) for x in logs if x.get("loss") is not None]
        if points:
            ax.plot([p[0] for p in points], [p[1] for p in points], marker=".")
            ax.set_xlabel("Step"); ax.set_ylabel("Training loss")
        else:
            ax.text(.5, .5, "No loss values logged", ha="center", va="center")
        ax.set_title("Central model training loss"); fig.tight_layout(); fig.savefig(loss_path); plt.close(fig)

        metadata = {
            "central_version": version, "language_added": language_name,
            "source_language": source_language, "target_language": target_language,
            "base_model": base_description, "rows_used": len(df),
            "train_rows": len(train_df), "held_out_rows": len(test_df),
            "training_steps": int(trainer.state.global_step), "bleu": bleu, "chrf": chrf,
            "elapsed_seconds": round(time.monotonic()-start_time, 2),
            "device": "cpu", "created_at": utc_now()
        }
        (model_dir / "central_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        pd.DataFrame([metadata]).to_csv(version_dir / "metrics.csv", index=False)
        pd.DataFrame(rows).to_csv(version_dir / "heldout_examples.csv", index=False, encoding="utf-8-sig")
        if METRICS_PATH.exists():
            pd.concat([pd.read_csv(METRICS_PATH), pd.DataFrame([metadata])], ignore_index=True).to_csv(METRICS_PATH, index=False)
        else:
            pd.DataFrame([metadata]).to_csv(METRICS_PATH, index=False)

        # Register language and point central state to the new saved version only after saving succeeded.
        reg = load_registry()
        entry = {
            "name": language_name, "code": str(lang_code or "").strip(),
            "source_language": source_language, "target_language": target_language,
            "central_version": version, "model_dir": str(model_dir),
            "created_at": metadata["created_at"], "status": "trained",
            "rows_used": len(df), "bleu": bleu, "chrf": chrf
        }
        old = [x for x in reg["languages"] if not (
            x.get("name", "").casefold() == language_name.casefold() and
            x.get("source_language", "").casefold() == source_language.casefold() and
            x.get("target_language", "").casefold() == target_language.casefold()
        )]
        old.append(entry); reg["languages"] = old; save_registry(reg)
        save_state({"central_model_dir": str(model_dir), "base_model_id": base_id, "version": version, "updated_at": utc_now()})

        zip_path = Path("/content") / f"central_multilingual_model_v{version}.zip"
        write_bundle(model_dir, reg, metadata, zip_path)
        elapsed = time.monotonic() - start_time
        status = (
            f"Central model updated and saved.\nVersion: {version}\n"
            f"Added: {language_name} ({source_language} → {target_language})\n"
            f"Dataset rows: {len(df)}; train: {len(train_df)}; held out: {len(test_df)}\n"
            f"Steps: {trainer.state.global_step}; time: {elapsed/60:.1f} min\n"
            f"BLEU: {bleu:.2f}; chrF: {chrf:.2f}\n"
            "The 18-minute budget applies to optimization approximately; loading, evaluation and saving add time."
        )
        return status, str(zip_path), str(loss_path), pd.DataFrame(rows), gr.update(choices=language_choices(), value=f"{source_language} → {target_language}")
    except Exception as e:
        return f"TRAINING FAILED: {type(e).__name__}: {e}\n{traceback.format_exc(limit=5)}", None, None, pd.DataFrame(), gr.update(choices=language_choices(), value="Add a language")

print("Central training pipeline defined.")

Central training pipeline defined.


## 5. Model-based inference (no example lookup)

In [5]:
_INFERENCE_CACHE = {}

def infer_language(text, selected_language):
    if not str(text or "").strip():
        return "Enter text to translate."
    if not selected_language or selected_language == "Add a language":
        return "Choose a trained translation direction, or use Add a language to train one."
    # Dropdown labels are directional, e.g. "Hindi → English".
    entries = []
    for item in load_registry()["languages"]:
        src = str(item.get("source_language", "")).strip()
        tgt = str(item.get("target_language", "")).strip()
        label = f"{src} → {tgt}" if src and tgt else str(item.get("name", "")).strip()
        if label == selected_language:
            entries.append(item)
    if not entries:
        return "This translation direction is not registered. Train it first or restore your model bundle."
    entry = entries[-1]
    state = load_state()
    model_dir = Path(state.get("central_model_dir") or entry.get("model_dir", ""))
    if not (model_dir / "config.json").exists():
        return "Central model files are missing. Restore your downloaded central-model ZIP first."
    try:
        key = str(model_dir)
        if key not in _INFERENCE_CACHE:
            tok = AutoTokenizer.from_pretrained(key, local_files_only=True)
            mdl = AutoModelForSeq2SeqLM.from_pretrained(key, local_files_only=True).to("cpu").eval()
            _INFERENCE_CACHE[key] = (tok, mdl)
        tok, mdl = _INFERENCE_CACHE[key]
        src = entry.get("source_language", "")
        tgt = entry.get("target_language", "")
        model_name = str(getattr(mdl.config, "_name_or_path", "")).lower()
        state_model_name = str(state.get("base_model_id", "")).lower()
        gen_kwargs = {"max_new_tokens": 128, "num_beams": 1}
        if "nllb" in model_name or "nllb" in state_model_name:
            src_code, tgt_code = source_code_for(src), source_code_for(tgt)
            if not src_code or not tgt_code:
                return f"Missing NLLB language-code mapping for {src} → {tgt}. Add the correct codes in source_code_for()."
            tok.src_lang = src_code
            tid = tok.convert_tokens_to_ids(tgt_code)
            if tid is None or tid == tok.unk_token_id:
                return f"Target language code {tgt_code} is not recognized by this tokenizer."
            gen_kwargs["forced_bos_token_id"] = int(tid)
        encoded = tok(str(text), return_tensors="pt", truncation=True, max_length=512)
        with torch.inference_mode():
            output = mdl.generate(**encoded, **gen_kwargs)
        return tok.batch_decode(output, skip_special_tokens=True)[0]
    except Exception as e:
        return f"TRANSLATION FAILED: {type(e).__name__}: {e}"

print("Inference function defined.")

Inference function defined.


## 6. Bundle restore helper

In [6]:
def restore_central_bundle(zip_path):
    try:
        if not zip_path:
            return "Choose a ZIP bundle."
        incoming = ROOT / "restore_incoming"
        if incoming.exists():
            shutil.rmtree(incoming)
        incoming.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            if any(Path(n).is_absolute() or ".." in Path(n).parts for n in names):
                raise ValueError("Unsafe ZIP path.")
            zf.extractall(incoming)
        model_dir = incoming / "central_model"
        if not (model_dir / "config.json").exists():
            raise ValueError("Bundle must contain central_model/config.json.")
        cfg = AutoConfig.from_pretrained(str(model_dir), local_files_only=True)
        if not getattr(cfg, "is_encoder_decoder", False):
            raise ValueError("Bundle model is not a seq2seq model.")
        reg_path = incoming / "languages.json"
        state_path = incoming / "central_state.json"
        if reg_path.exists():
            obj = json.loads(reg_path.read_text(encoding="utf-8"))
            save_registry(obj)
        version = 1
        try:
            meta = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
            version = int(meta.get("central_version", 1))
        except Exception:
            meta = {}
        # Copy into durable-in-session version folder; update central pointer only after validation.
        dest = VERSIONS_DIR / f"restored_v{version}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        shutil.copytree(model_dir, dest)
        state = load_state()
        state.update({"central_model_dir": str(dest), "version": version, "updated_at": utc_now()})
        save_state(state)
        _INFERENCE_CACHE.clear()
        return f"Central model restored successfully from ZIP. Version: {version}. Languages: {', '.join(x.get('name','') for x in load_registry()['languages']) or 'none'}"
    except Exception as e:
        return f"RESTORE FAILED: {type(e).__name__}: {e}"

print("Bundle restore helper defined.")

Bundle restore helper defined.


## 7. Quick non-network smoke tests

In [7]:
# No model download here: exercise auto-column detection and registry serialization.
_smoke = ROOT / "_smoke.csv"
pd.DataFrame({
    "text_in_language": ["नमस्ते", "धन्यवाद", "आप कैसे हैं?", "शुभ रात्रि", "कृपया", "मैं ठीक हूँ", "फिर मिलेंगे", "स्वागत है"],
    "translated_text": ["Hello", "Thank you", "How are you?", "Good night", "Please", "I am fine", "See you", "Welcome"]
}).to_csv(_smoke, index=False)
_smoke_df, _smoke_src, _smoke_tgt = clean_parallel_data(str(_smoke), 100)
assert len(_smoke_df) == 8
assert (_smoke_src, _smoke_tgt) == ("text_in_language", "translated_text")
_smoke.unlink(missing_ok=True)
assert source_code_for("Hindi") == "hin_Deva"
assert source_code_for("Marathi") == "mar_Deva"
print("Smoke tests passed: dataset auto-detection and known language-code mapping.")
print("Model download/training/inference still require an end-to-end Colab run.")

Smoke tests passed: dataset auto-detection and known language-code mapping.
Model download/training/inference still require an end-to-end Colab run.


## 8. Minimal interface

In [8]:
def toggle_add_panel(choice):
    return gr.update(visible=(choice == "Add a language"))

def refresh_languages():
    choices = language_choices()
    value = choices[0] if len(choices) > 1 else "Add a language"
    return gr.update(choices=choices, value=value)

with gr.Blocks(title="Central Multilingual Translator") as app:
    gr.Markdown("# Central Multilingual Translator")
    _initial_choices = language_choices()
    _initial_value = _initial_choices[0] if len(_initial_choices) > 1 else "Add a language"
    language = gr.Dropdown(label="Languages / translation direction", choices=_initial_choices, value=_initial_value, interactive=True)
    with gr.Group(visible=(_initial_value == "Add a language")) as add_panel:
        gr.Markdown("### Add a language")
        gr.Markdown("Upload a parallel dataset to train, and optionally a compatible model ZIP to continue from that central model.")
        dataset_file = gr.File(label="Dataset file (CSV/XLSX)", file_types=[".csv", ".xlsx", ".xls"], type="filepath")
        model_zip = gr.File(label="Optional existing central-model ZIP", file_types=[".zip"], type="filepath")
        with gr.Row():
            language_name = gr.Textbox(label="Language name", placeholder="Hindi")
            language_code = gr.Textbox(label="Language code (optional)", placeholder="hi")
        with gr.Row():
            source_language = gr.Textbox(label="Source language", value="Hindi")
            target_language = gr.Textbox(label="Target language", value="English")
        max_rows = gr.Slider(100, 1000, value=500, step=100, label="Max dataset rows (RAM-safe CPU default)")
        max_length = gr.Slider(32, 96, value=64, step=16, label="Max tokens (lower uses less RAM)")
        epochs = gr.Slider(1, 3, value=1, step=1, label="Epochs")
        budget = gr.Slider(1, 18, value=18, step=1, label="Training budget (minutes)")
        train_btn = gr.Button("Train / extend central model")
        train_status = gr.Textbox(label="Training status", lines=6)
        download_bundle = gr.File(label="Download updated central model ZIP")
        loss_graph = gr.Image(label="Training loss graph")
        heldout = gr.Dataframe(label="Held-out translation examples (up to 8; CPU/RAM limited)", interactive=False)

        train_btn.click(
            fn=train_or_extend_central,
            inputs=[dataset_file, language_name, source_language, target_language, language_code,
                    max_rows, max_length, epochs, budget, model_zip],
            outputs=[train_status, download_bundle, loss_graph, heldout, language]
        )

    original = gr.Textbox(label="Original text", lines=4, placeholder="Enter text to translate")
    translated = gr.Textbox(label="Translate text", lines=4, interactive=False)
    translate_btn = gr.Button("Translate")
    refresh_btn = gr.Button("Refresh languages")

    def translate_selected(text, choice):
        if choice == "Add a language":
            return "Choose a learned language, or train one using Add a language."
        return infer_language(text, choice)

    translate_btn.click(fn=translate_selected, inputs=[original, language], outputs=translated)
    language.change(fn=toggle_add_panel, inputs=language, outputs=add_panel)
    refresh_btn.click(fn=refresh_languages, inputs=[], outputs=language)

    with gr.Accordion("Restore downloaded central model ZIP", open=False):
        restore_file = gr.File(label="Central model ZIP", file_types=[".zip"], type="filepath")
        restore_btn = gr.Button("Restore model bundle")
        restore_status = gr.Textbox(label="Restore status")
        restore_btn.click(fn=restore_central_bundle, inputs=[restore_file], outputs=restore_status)

print("Minimal interface built: no Base translation model ID or manual dataset-column controls.")

Minimal interface built: no Base translation model ID or manual dataset-column controls.


## 9. Launch

## 9. Interface behavior check
The Add a language panel is visible on first launch when no language pairs are registered. After training, the first trained direction is selected by default. Translation directions are displayed explicitly to avoid ambiguity.

In [9]:
# Static checks for the initial UI state and directional dropdown logic.
_initial_choices_check = language_choices()
_initial_value_check = _initial_choices_check[0] if len(_initial_choices_check) > 1 else "Add a language"
assert _initial_value_check in _initial_choices_check
assert ("Add a language" in _initial_choices_check)
print("UI checks passed. Initial selection:", _initial_value_check)
print("Available choices:", _initial_choices_check)

UI checks passed. Initial selection: Add a language
Available choices: ['Add a language']


In [10]:
app.launch(share=False, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## Important usage notes
- First use requires internet access to download the multilingual base checkpoint. The default NLLB checkpoint is relatively large and CPU inference/training may be slow or memory-intensive.
- Dataset direction must match the source/target language fields. For NLLB, language names must be supported by `source_code_for()`; add the proper NLLB tokenizer code for other languages.
- For a language already registered, the central model is used. Current UI records each language pair; it does not automatically detect arbitrary source/target language pairs from the input.
- The uploaded ZIP must contain `central_model/config.json` (the bundle produced here) or a compatible model directory at the root/immediate subdirectory. A model ZIP is used as the central starting point; a parallel dataset is still required for fine-tuning.
- Download the generated ZIP before ending the Colab session. `/content` is ephemeral.
- Training time limit is approximate. Training can stop after the budget, but setup/evaluation/save overhead may exceed 20 minutes.
- Quality metrics (BLEU and chrF) are computed on a small held-out sample. They are useful for smoke testing, not definitive research evaluation. Use a larger, human-verified test set for a paper.

**Direction selection:** the dropdown displays explicit pairs (for example, `Hindi → English`) so that adding both directions does not make inference silently pick the wrong pair. The registry still stores the language names and both directions separately.